## 3-1. QURI Parts とは

古典コンピュータでは，機械語のような低レイヤーのプログラミングと，人間が理解しやすい高レイヤーのプログラミングとを分けて考えることが自然である。量子コンピュータでも同様の階層構造を考えることができるが，現在の量子コンピュータは量子ビット数やノイズの制約が厳しく，ハードウェア依存の低レイヤーな操作を意識せざるをえない場面が多い。

その結果，IBM 系では Qiskit，Google 系では Cirq，シミュレータでは Qulacs というように，実行環境ごとにコードを書き分ける必要が生じやすい。こうした移植コストを減らすために開発されたのが，QunaSys 社による Python ライブラリ **QURI Parts** である。QURI Parts は，量子状態，量子ゲート，量子回路，オブザーバブルなどに対する統一的なインターフェースを提供し，複数のバックエンドへ橋渡しする役割を担う。

[図 3.1 プレースホルダ: QURI Parts の位置づけ]

まずは以下のようにインストールする。

```python
pip install quri-parts
pip install "quri-parts[qulacs]"
```

### 量子状態

QURI Parts では `quantum_state()` を用いて量子状態を作成できる。計算基底状態のビット列を指定する方法，状態ベクトルを直接指定する方法，量子回路を初期状態に作用させた状態として定義する方法がある。bits を使うときは，0 番目の量子ビットが二進表記の最下位ビットに対応する点に注意する。

In [ ]:
from quri_parts.core import quantum_state, QuantumCircuit
from quri_parts.circuit.gates import H, CNOT

n = 5
state = quantum_state(n)
comp_state = quantum_state(n, bits=0b10100)

circuit = QuantumCircuit(n, gates=[H(0), CNOT(0, 1)])
state_with_circuit = quantum_state(n, bits=0b10100, circuit=circuit)
comp_state

`qubit_count` で量子ビット数を確認でき，`evaluate_state_to_vector()` を使うと状態ベクトルを取得できる。ただし後者は大きな量子ビット数では非常に重い操作である。状態の重なりやサンプリングも QURI Parts 上で扱うことができる。

In [ ]:
from quri_parts.qulacs import evaluate_state_to_vector
from quri_parts.qulacs.overlap_estimator import create_qulacs_vector_overlap_estimator

small_state = quantum_state(2, circuit=QuantumCircuit(2, gates=[H(0)]))
vector = evaluate_state_to_vector(small_state)

overlap_estimator = create_qulacs_vector_overlap_estimator()
reference = quantum_state(2)
vector, overlap_estimator(small_state, reference).value

### 量子ゲートと量子回路

QURI Parts にはパウリゲート，Clifford ゲート，回転ゲート，パウリ回転ゲートなどが用意されている。量子回路は `QuantumCircuit` として表現され，ゲートの追加，回路深さの確認，結合や拡張，可視化，バックエンド向けの回路変換などが可能である。

In [ ]:
import math
from quri_parts.circuit import QuantumCircuit, X
from quri_parts.qulacs.circuit import convert_circuit

circuit = QuantumCircuit(3)
circuit.add_gate(X(0))
circuit.add_RX_gate(1, math.pi / 3)
circuit.add_CNOT_gate(2, 1)
circuit.add_PauliRotation_gate(target_qubits=(0, 1, 2), pauli_id_list=(1, 2, 3), angle=math.pi / 3)

print(circuit.qubit_count, circuit.depth)
print(circuit.gates)
convert_circuit(circuit)

[図 3.2 プレースホルダ: depth が 2 の量子回路]

また，QURI Parts では回路の可視化やトランスパイルも可能である。たとえば `draw_circuit()` を使えばアスキーアートで回路を確認でき，`CliffordRZSetTranspiler` などを用いて等価な回路へ変換できる。

### オブザーバブルと期待値推定

量子力学では物理量はエルミート演算子で表され，QURI Parts ではパウリ演算子の線形結合として `Operator` を用いて表現する。`pauli_label()` で `PauliLabel` を作り，係数付き辞書から `Operator` を作成できる。これらに対して加減算，スカラー倍，積，エルミート共役などの操作が定義されている。

In [ ]:
from quri_parts.core.operator import Operator, PAULI_IDENTITY, pauli_label
from quri_parts.qulacs.estimator import create_qulacs_vector_estimator

op = Operator({
    PAULI_IDENTITY: 8,
    pauli_label("X0 Y1"): 2,
    pauli_label("Z0 Y1"): 2j,
})

state = quantum_state(
    4,
    circuit=QuantumCircuit(4, gates=[X(0), H(1), H(2), CNOT(1, 2)])
)

estimator = create_qulacs_vector_estimator()
op, estimator(op, state)

### 変分量子状態

量子アルゴリズムの中には，変分パラメータをもつ量子状態を用いるものがある。QURI Parts では `UnboundParametricQuantumCircuit` や `LinearMappedUnboundParametricQuantumCircuit` により，パラメータ付き回路を効率よく扱える。`Parameter` は具体値を持たないプレースホルダであり，同じ名前でも別オブジェクトなら異なるパラメータとして扱われる。

In [ ]:
from quri_parts.circuit import UnboundParametricQuantumCircuit

parametric_circuit = UnboundParametricQuantumCircuit(2)
parametric_circuit.add_H_gate(0)
parametric_circuit.add_CNOT_gate(0, 1)
p_theta = parametric_circuit.add_ParametricRX_gate(0)
p_phi = parametric_circuit.add_ParametricRY_gate(0)
p_psi = parametric_circuit.add_ParametricRZ_gate(1)

p_theta, p_phi, p_psi, parametric_circuit